# Credit Card Default Prediction — Modeling

This notebook implements the modeling phase for the credit card default prediction project. Based on the EDA findings, we build and evaluate multiple classifiers across two feature settings, optimize decision thresholds, and analyze feature importance.

## Outline

| Step | Description |
|------|-------------|
| 1 | Data Preprocessing — encoding fix, feature engineering, train/test split, scaling |
| 2 | Baseline Models — Logistic Regression, Decision Tree, KNN, SVM |
| 3 | Stronger Models — Random Forest, Gradient Boosting, XGBoost |
| 4 | Feature Set Comparison — raw features vs. raw + engineered behavior features |
| 5 | Evaluation Metrics — ROC curve, PR curve, confusion matrix, training time |
| 6 | Threshold Optimization — risk-sensitive decision threshold tuning |
| 7 | Interpretability — feature importance and coefficient analysis |
| 8 | Summary — final model ranking and key findings |


In [1]:
import matplotlib
matplotlib.use('Agg')

import os
import re
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

import xgboost as xgb

from IPython.display import display

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:.4f}'.format)

# =========================
# Paths
# =========================

DATA_PATH = 'EDA/UCI_Credit_Card.xls'

OUTPUT_DIR = Path('../outputs')
FIG_DIR = OUTPUT_DIR / 'modeling_figures'
RESULT_DIR = OUTPUT_DIR / 'results'

OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Plot style
# =========================

sns.set_theme(style='white', context='talk')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 15
plt.rcParams['xtick.labelsize'] = 13
plt.rcParams['ytick.labelsize'] = 13
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['font.family'] = 'DejaVu Sans'

# =========================
# Helpers
# =========================

def save_fig(filename):
    path = FIG_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'Saved figure: {path}')

def save_csv(df, filename):
    path = RESULT_DIR / filename
    df.to_csv(path, index=False)
    print(f'Saved CSV: {path}')

# =========================
# Load data
# =========================

df = pd.read_csv(DATA_PATH, encoding='utf-8')

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

Dataset loaded successfully.
Shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0000,2,2,1,24,2,2,-1,-1,-2,-2,3913.0000,3102.0000,689.0000,0.0000,0.0000,0.0000,0.0000,689.0000,0.0000,0.0000,0.0000,0.0000,1
1,2,120000.0000,2,2,2,26,-1,2,0,0,0,2,2682.0000,1725.0000,2682.0000,3272.0000,3455.0000,3261.0000,0.0000,1000.0000,1000.0000,1000.0000,0.0000,2000.0000,1
2,3,90000.0000,2,2,2,34,0,0,0,0,0,0,29239.0000,14027.0000,13559.0000,14331.0000,14948.0000,15549.0000,1518.0000,1500.0000,1000.0000,1000.0000,1000.0000,5000.0000,0
3,4,50000.0000,2,2,1,37,0,0,0,0,0,0,46990.0000,48233.0000,49291.0000,28314.0000,28959.0000,29547.0000,2000.0000,2019.0000,1200.0000,1100.0000,1069.0000,1000.0000,0
4,5,50000.0000,1,2,1,57,-1,0,-1,0,0,0,8617.0000,5670.0000,35835.0000,20940.0000,19146.0000,19131.0000,2000.0000,36681.0000,10000.0000,9000.0000,689.0000,679.0000,0


## Step 1: Data Preprocessing

We apply three preprocessing steps before model training:
1. **Categorical encoding fix** — EDUCATION contains non-standard values (0, 5, 6); MARRIAGE contains 0
2. **Behavior feature engineering** — create 17 repayment-behavior features as identified in EDA
3. **Train/test split + scaling** — stratified 80/20 split, StandardScaler for feature normalization


In [2]:
target_col = 'default.payment.next.month'

df_model = df.copy()

df_model['EDUCATION'] = df_model['EDUCATION'].apply(
    lambda x: 4 if x in [0, 5, 6] else x
)

df_model['MARRIAGE'] = df_model['MARRIAGE'].apply(
    lambda x: 3 if x == 0 else x
)

print('EDUCATION categories after fix:', sorted(df_model['EDUCATION'].unique()))
print('MARRIAGE categories after fix:', sorted(df_model['MARRIAGE'].unique()))

edu_summary = df_model['EDUCATION'].value_counts().sort_index().to_frame('count')
marriage_summary = df_model['MARRIAGE'].value_counts().sort_index().to_frame('count')
display(edu_summary)
display(marriage_summary)
save_csv(edu_summary.reset_index(), 'education_after_fix.csv')
save_csv(marriage_summary.reset_index(), 'marriage_after_fix.csv')

EDUCATION categories after fix: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
MARRIAGE categories after fix: [np.int64(1), np.int64(2), np.int64(3)]


,count
EDUCATION,
1,10585
2,14030
3,4917
4,468


,count
MARRIAGE,
1,13659
2,15964
3,377


Saved CSV: ..\outputs\results\education_after_fix.csv
Saved CSV: ..\outputs\results\marriage_after_fix.csv


In [3]:
pay_status_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
pay_amt_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

df_features = df_model.copy()

# --- Repayment delay behavior ---
df_features['delay_count'] = (df_features[pay_status_cols] > 0).sum(axis=1)
df_features['severe_delay_count'] = (df_features[pay_status_cols] >= 2).sum(axis=1)
df_features['max_delay'] = df_features[pay_status_cols].max(axis=1)
df_features['avg_delay'] = df_features[pay_status_cols].mean(axis=1)
df_features['recent_delay'] = df_features['PAY_0']
df_features['recent_delay_flag'] = (df_features['PAY_0'] > 0).astype(int)

# --- Bill amount behavior ---
df_features['avg_bill_amt'] = df_features[bill_cols].mean(axis=1)
df_features['max_bill_amt'] = df_features[bill_cols].max(axis=1)
df_features['recent_bill_amt'] = df_features['BILL_AMT1']
df_features['bill_trend'] = df_features['BILL_AMT1'] - df_features['BILL_AMT6']
df_features['credit_utilization'] = df_features['BILL_AMT1'] / (df_features['LIMIT_BAL'].replace(0, np.nan))

# --- Payment amount behavior ---
df_features['avg_pay_amt'] = df_features[pay_amt_cols].mean(axis=1)
df_features['max_pay_amt'] = df_features[pay_amt_cols].max(axis=1)
df_features['recent_pay_amt'] = df_features['PAY_AMT1']
df_features['pay_trend'] = df_features['PAY_AMT1'] - df_features['PAY_AMT6']

# --- Payment-to-bill ratios ---
df_features['recent_pay_bill_ratio'] = df_features['PAY_AMT1'] / (df_features['BILL_AMT2'].abs() + 1)
df_features['total_pay_bill_ratio'] = (df_features[pay_amt_cols].sum(axis=1) / (df_features[bill_cols].abs().sum(axis=1) + 1))

df_features = df_features.replace([np.inf, -np.inf], np.nan)

engineered_features = [
    'delay_count', 'severe_delay_count', 'max_delay', 'avg_delay',
    'recent_delay', 'recent_delay_flag', 'avg_bill_amt', 'max_bill_amt',
    'recent_bill_amt', 'bill_trend', 'credit_utilization', 'avg_pay_amt',
    'max_pay_amt', 'recent_pay_amt', 'pay_trend',
    'recent_pay_bill_ratio', 'total_pay_bill_ratio'
]

print('Engineered features created:', len(engineered_features))

nan_counts = df_features[engineered_features].isnull().sum()
if nan_counts.sum() > 0:
    print('\nNaN values found:')
    print(nan_counts[nan_counts > 0])
    for col in nan_counts[nan_counts > 0].index:
        df_features[col] = df_features[col].fillna(df_features[col].median())
    print('NaN values filled with median.')
else:
    print('\nNo NaN values in engineered features.')

Engineered features created: 17

No NaN values in engineered features.


In [4]:
target = target_col

raw_features = [
    'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

feature_set_a = raw_features.copy()
feature_set_b = raw_features + engineered_features

print(f'Set A (raw features only): {len(feature_set_a)} features')
print(f'Set B (raw + engineered):  {len(feature_set_b)} features')

X_a = df_features[feature_set_a].copy()
X_b = df_features[feature_set_b].copy()
y = df_features[target].copy()

X_a_train, X_a_test, y_train, y_test = train_test_split(
    X_a, y, test_size=0.2, random_state=42, stratify=y
)
X_b_train, X_b_test, _, _ = train_test_split(
    X_b, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTraining set size: {len(y_train)} ({y_train.mean()*100:.1f}% default)')
print(f'Test set size:     {len(y_test)} ({y_test.mean()*100:.1f}% default)')

scaler_a = StandardScaler()
scaler_b = StandardScaler()

X_a_train_s = pd.DataFrame(scaler_a.fit_transform(X_a_train), columns=X_a_train.columns, index=X_a_train.index)
X_a_test_s = pd.DataFrame(scaler_a.transform(X_a_test), columns=X_a_test.columns, index=X_a_test.index)
X_b_train_s = pd.DataFrame(scaler_b.fit_transform(X_b_train), columns=X_b_train.columns, index=X_b_train.index)
X_b_test_s = pd.DataFrame(scaler_b.transform(X_b_test), columns=X_b_test.columns, index=X_b_test.index)

print('\nFeature scaling complete (StandardScaler).')

Set A (raw features only): 23 features
Set B (raw + engineered):  40 features

Training set size: 24000 (22.1% default)
Test set size:     6000 (22.1% default)

Feature scaling complete (StandardScaler).


## Steps 2 & 3: Model Training and Evaluation

We train **7 classifiers** × **2 feature sets** = **14 model evaluations**:

| Model | Type | Key Parameters |
|-------|------|----------------|
| Logistic Regression | Baseline | `class_weight="balanced"`, `max_iter=1000` |
| Decision Tree | Baseline | `max_depth=8`, `class_weight="balanced"` |
| KNN | Baseline | `n_neighbors=5` |
| SVM | Baseline | `LinearSVC + CalibratedClassifierCV` |
| Random Forest | Ensemble | `n_estimators=200`, `max_depth=12` |
| Gradient Boosting | Ensemble | `n_estimators=200`, `max_depth=5` |
| XGBoost | Ensemble | `scale_pos_weight` for imbalance |

Each model is evaluated on: Accuracy, Precision, Recall, F1-score, ROC-AUC, PR-AUC, and Training Time.


In [5]:
def evaluate_model(model, model_name, X_train, X_test, y_train, y_test, feature_set_name, results_list):
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)

    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_proba = model.decision_function(X_test)
    else:
        y_proba = None

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    if y_proba is not None:
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    result = {
        'Model': model_name,
        'Feature Set': feature_set_name,
        'Accuracy': round(accuracy, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-score': round(f1, 4),
        'ROC-AUC': round(roc_auc, 4) if not np.isnan(roc_auc) else 'N/A',
        'PR-AUC': round(pr_auc, 4) if not np.isnan(pr_auc) else 'N/A',
        'Train Time (s)': round(train_time, 3),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)
    }
    results_list.append(result)

    print(f'{model_name:25s} | {feature_set_name:6s} | Acc={accuracy:.4f} | Prec={precision:.4f} | Rec={recall:.4f} | F1={f1:.4f} | ROC={roc_auc:.4f} | PR={pr_auc:.4f} | Time={train_time:.3f}s')
    return model

all_results = []
trained_models = {}

In [6]:
print('=' * 80)
print('LOGISTIC REGRESSION')
print('=' * 80)

lr_a = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
evaluate_model(lr_a, 'Logistic Regression', X_a_train_s, X_a_test_s, y_train, y_test, 'Set A', all_results)
trained_models[('Logistic Regression', 'Set A')] = lr_a

lr_b = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
evaluate_model(lr_b, 'Logistic Regression', X_b_train_s, X_b_test_s, y_train, y_test, 'Set B', all_results)
trained_models[('Logistic Regression', 'Set B')] = lr_b

save_csv(pd.DataFrame([r for r in all_results if r['Model'] == 'Logistic Regression']), 'logistic_regression_results.csv')

LOGISTIC REGRESSION
Logistic Regression       | Set A  | Acc=0.6795 | Prec=0.3671 | Rec=0.6202 | F1=0.4612 | ROC=0.7084 | PR=0.4908 | Time=0.038s
Logistic Regression       | Set B  | Acc=0.7558 | Prec=0.4599 | Rec=0.5968 | F1=0.5195 | ROC=0.7583 | PR=0.5243 | Time=0.053s
Saved CSV: ..\outputs\results\logistic_regression_results.csv


In [7]:
print('=' * 80)
print('DECISION TREE')
print('=' * 80)

dt_a = DecisionTreeClassifier(random_state=42, max_depth=8, min_samples_leaf=10, class_weight='balanced')
evaluate_model(dt_a, 'Decision Tree', X_a_train, X_a_test, y_train, y_test, 'Set A', all_results)
trained_models[('Decision Tree', 'Set A')] = dt_a

dt_b = DecisionTreeClassifier(random_state=42, max_depth=8, min_samples_leaf=10, class_weight='balanced')
evaluate_model(dt_b, 'Decision Tree', X_b_train, X_b_test, y_train, y_test, 'Set B', all_results)
trained_models[('Decision Tree', 'Set B')] = dt_b

save_csv(pd.DataFrame([r for r in all_results if r['Model'] == 'Decision Tree']), 'decision_tree_results.csv')

DECISION TREE
Decision Tree             | Set A  | Acc=0.7235 | Prec=0.4188 | Rec=0.6451 | F1=0.5079 | ROC=0.7529 | PR=0.5136 | Time=0.163s
Decision Tree             | Set B  | Acc=0.7317 | Prec=0.4307 | Rec=0.6624 | F1=0.5220 | ROC=0.7646 | PR=0.5288 | Time=0.308s
Saved CSV: ..\outputs\results\decision_tree_results.csv


In [8]:
print('=' * 80)
print('K-NEAREST NEIGHBORS')
print('=' * 80)

knn_a = KNeighborsClassifier(n_neighbors=5)
evaluate_model(knn_a, 'KNN', X_a_train_s, X_a_test_s, y_train, y_test, 'Set A', all_results)
trained_models[('KNN', 'Set A')] = knn_a

knn_b = KNeighborsClassifier(n_neighbors=5)
evaluate_model(knn_b, 'KNN', X_b_train_s, X_b_test_s, y_train, y_test, 'Set B', all_results)
trained_models[('KNN', 'Set B')] = knn_b

save_csv(pd.DataFrame([r for r in all_results if r['Model'] == 'KNN']), 'knn_results.csv')

K-NEAREST NEIGHBORS
KNN                       | Set A  | Acc=0.7923 | Prec=0.5474 | Rec=0.3527 | F1=0.4290 | ROC=0.7015 | PR=0.4137 | Time=0.003s
KNN                       | Set B  | Acc=0.7928 | Prec=0.5473 | Rec=0.3662 | F1=0.4388 | ROC=0.7075 | PR=0.4222 | Time=0.006s
Saved CSV: ..\outputs\results\knn_results.csv


In [9]:
print('=' * 80)
print('SUPPORT VECTOR MACHINE')
print('=' * 80)

svm_a = CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=3000, random_state=42))
evaluate_model(svm_a, 'SVM', X_a_train_s, X_a_test_s, y_train, y_test, 'Set A', all_results)
trained_models[('SVM', 'Set A')] = svm_a

svm_b = CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=3000, random_state=42))
evaluate_model(svm_b, 'SVM', X_b_train_s, X_b_test_s, y_train, y_test, 'Set B', all_results)
trained_models[('SVM', 'Set B')] = svm_b

save_csv(pd.DataFrame([r for r in all_results if r['Model'] == 'SVM']), 'svm_results.csv')

SUPPORT VECTOR MACHINE
SVM                       | Set A  | Acc=0.8083 | Prec=0.6928 | Rec=0.2396 | F1=0.3561 | ROC=0.7077 | PR=0.4900 | Time=0.224s
SVM                       | Set B  | Acc=0.8162 | Prec=0.6564 | Rec=0.3542 | F1=0.4601 | ROC=0.7559 | PR=0.5173 | Time=0.657s
Saved CSV: ..\outputs\results\svm_results.csv


In [10]:
print('=' * 80)
print('RANDOM FOREST')
print('=' * 80)

rf_a = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)
evaluate_model(rf_a, 'Random Forest', X_a_train, X_a_test, y_train, y_test, 'Set A', all_results)
trained_models[('Random Forest', 'Set A')] = rf_a

rf_b = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)
evaluate_model(rf_b, 'Random Forest', X_b_train, X_b_test, y_train, y_test, 'Set B', all_results)
trained_models[('Random Forest', 'Set B')] = rf_b

save_csv(pd.DataFrame([r for r in all_results if r['Model'] == 'Random Forest']), 'random_forest_results.csv')

RANDOM FOREST
Random Forest             | Set A  | Acc=0.7907 | Prec=0.5250 | Rec=0.5614 | F1=0.5426 | ROC=0.7752 | PR=0.5504 | Time=0.797s
Random Forest             | Set B  | Acc=0.7922 | Prec=0.5285 | Rec=0.5584 | F1=0.5431 | ROC=0.7785 | PR=0.5545 | Time=1.090s
Saved CSV: ..\outputs\results\random_forest_results.csv


In [11]:
print('=' * 80)
print('GRADIENT BOOSTING')
print('=' * 80)

gb_a = GradientBoostingClassifier(n_estimators=200, max_depth=5, min_samples_leaf=10, learning_rate=0.1, random_state=42)
evaluate_model(gb_a, 'Gradient Boosting', X_a_train, X_a_test, y_train, y_test, 'Set A', all_results)
trained_models[('Gradient Boosting', 'Set A')] = gb_a

gb_b = GradientBoostingClassifier(n_estimators=200, max_depth=5, min_samples_leaf=10, learning_rate=0.1, random_state=42)
evaluate_model(gb_b, 'Gradient Boosting', X_b_train, X_b_test, y_train, y_test, 'Set B', all_results)
trained_models[('Gradient Boosting', 'Set B')] = gb_b

save_csv(pd.DataFrame([r for r in all_results if r['Model'] == 'Gradient Boosting']), 'gradient_boosting_results.csv')

GRADIENT BOOSTING
Gradient Boosting         | Set A  | Acc=0.8158 | Prec=0.6529 | Rec=0.3572 | F1=0.4618 | ROC=0.7722 | PR=0.5408 | Time=20.869s
Gradient Boosting         | Set B  | Acc=0.8180 | Prec=0.6594 | Rec=0.3662 | F1=0.4709 | ROC=0.7778 | PR=0.5507 | Time=40.752s
Saved CSV: ..\outputs\results\gradient_boosting_results.csv


In [12]:
print('=' * 80)
print('XGBOOST')
print('=' * 80)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_a = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, verbosity=0)
evaluate_model(xgb_a, 'XGBoost', X_a_train, X_a_test, y_train, y_test, 'Set A', all_results)
trained_models[('XGBoost', 'Set A')] = xgb_a

xgb_b = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, verbosity=0)
evaluate_model(xgb_b, 'XGBoost', X_b_train, X_b_test, y_train, y_test, 'Set B', all_results)
trained_models[('XGBoost', 'Set B')] = xgb_b

save_csv(pd.DataFrame([r for r in all_results if r['Model'] == 'XGBoost']), 'xgboost_results.csv')

XGBOOST
XGBoost                   | Set A  | Acc=0.7632 | Prec=0.4715 | Rec=0.5855 | F1=0.5224 | ROC=0.7653 | PR=0.5425 | Time=0.450s
XGBoost                   | Set B  | Acc=0.7668 | Prec=0.4785 | Rec=0.6051 | F1=0.5344 | ROC=0.7722 | PR=0.5465 | Time=0.457s
Saved CSV: ..\outputs\results\xgboost_results.csv


## Step 4: Feature Set Comparison

We compare **Set A (raw features only)** vs **Set B (raw + engineered behavior features)** across all models. A positive delta (B - A) indicates that engineered features added predictive value.


In [13]:
results_df = pd.DataFrame(all_results)

display_cols = ['Model', 'Feature Set', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'PR-AUC', 'Train Time (s)']
display_results = results_df[display_cols].copy()

print('=' * 120)
print('COMPLETE MODEL COMPARISON: SET A vs SET B')
print('=' * 120)
display(display_results)
save_csv(display_results, 'all_models_comparison.csv')

comparison_rows = []
for model_name in results_df['Model'].unique():
    row = {'Model': model_name}
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'PR-AUC']:
        set_a_val = results_df.loc[(results_df['Model'] == model_name) & (results_df['Feature Set'] == 'Set A'), metric].values[0]
        set_b_val = results_df.loc[(results_df['Model'] == model_name) & (results_df['Feature Set'] == 'Set B'), metric].values[0]
        row[f'{metric} (A)'] = set_a_val
        row[f'{metric} (B)'] = set_b_val
        row[f'{metric} Delta'] = set_b_val - set_a_val
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
print('\nFeature Set Comparison: Set B - Set A (positive = improvement)')
display(comparison_df)
save_csv(comparison_df, 'feature_set_comparison.csv')

COMPLETE MODEL COMPARISON: SET A vs SET B


,Model,Feature Set,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC,Train Time (s)
0,Logistic Regression,Set A,0.6795,0.3671,0.6202,0.4612,0.7084,0.4908,0.0380
1,Logistic Regression,Set B,0.7558,0.4599,0.5968,0.5195,0.7583,0.5243,0.0530
2,Decision Tree,Set A,0.7235,0.4188,0.6451,0.5079,0.7529,0.5136,0.1630
3,Decision Tree,Set B,0.7317,0.4307,0.6624,0.5220,0.7646,0.5288,0.3080
4,KNN,Set A,0.7923,0.5474,0.3527,0.4290,0.7015,0.4137,0.0030
5,KNN,Set B,0.7928,0.5473,0.3662,0.4388,0.7075,0.4222,0.0060
6,SVM,Set A,0.8083,0.6928,0.2396,0.3561,0.7077,0.4900,0.2240
7,SVM,Set B,0.8162,0.6564,0.3542,0.4601,0.7559,0.5173,0.6570
8,Random Forest,Set A,0.7907,0.5250,0.5614,0.5426,0.7752,0.5504,0.7970
9,Random Forest,Set B,0.7922,0.5285,0.5584,0.5431,0.7785,0.5545,1.0900


Saved CSV: ..\outputs\results\all_models_comparison.csv

Feature Set Comparison: Set B - Set A (positive = improvement)


,Model,Accuracy (A),Accuracy (B),Accuracy Delta,Precision (A),Precision (B),Precision Delta,Recall (A),Recall (B),Recall Delta,F1-score (A),F1-score (B),F1-score Delta,ROC-AUC (A),ROC-AUC (B),ROC-AUC Delta,PR-AUC (A),PR-AUC (B),PR-AUC Delta
0,Logistic Regression,0.6795,0.7558,0.0763,0.3671,0.4599,0.0928,0.6202,0.5968,-0.0234,0.4612,0.5195,0.0583,0.7084,0.7583,0.0499,0.4908,0.5243,0.0335
1,Decision Tree,0.7235,0.7317,0.0082,0.4188,0.4307,0.0119,0.6451,0.6624,0.0173,0.5079,0.5220,0.0141,0.7529,0.7646,0.0117,0.5136,0.5288,0.0152
2,KNN,0.7923,0.7928,0.0005,0.5474,0.5473,-0.0001,0.3527,0.3662,0.0135,0.4290,0.4388,0.0098,0.7015,0.7075,0.0060,0.4137,0.4222,0.0085
3,SVM,0.8083,0.8162,0.0079,0.6928,0.6564,-0.0364,0.2396,0.3542,0.1146,0.3561,0.4601,0.1040,0.7077,0.7559,0.0482,0.4900,0.5173,0.0273
4,Random Forest,0.7907,0.7922,0.0015,0.5250,0.5285,0.0035,0.5614,0.5584,-0.0030,0.5426,0.5431,0.0005,0.7752,0.7785,0.0033,0.5504,0.5545,0.0041
5,Gradient Boosting,0.8158,0.8180,0.0022,0.6529,0.6594,0.0065,0.3572,0.3662,0.0090,0.4618,0.4709,0.0091,0.7722,0.7778,0.0056,0.5408,0.5507,0.0099
6,XGBoost,0.7632,0.7668,0.0036,0.4715,0.4785,0.0070,0.5855,0.6051,0.0196,0.5224,0.5344,0.0120,0.7653,0.7722,0.0069,0.5425,0.5465,0.0040


Saved CSV: ..\outputs\results\feature_set_comparison.csv


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor='white')

for ax_idx, metric in enumerate(['F1-score', 'ROC-AUC']):
    ax = axes[ax_idx]
    models_sorted = results_df['Model'].unique()
    x = np.arange(len(models_sorted))
    width = 0.35

    set_a_vals = [results_df.loc[(results_df['Model'] == m) & (results_df['Feature Set'] == 'Set A'), metric].values[0] for m in models_sorted]
    set_b_vals = [results_df.loc[(results_df['Model'] == m) & (results_df['Feature Set'] == 'Set B'), metric].values[0] for m in models_sorted]

    bars_a = ax.bar(x - width/2, set_a_vals, width, label='Set A (Raw Only)', color='#C9D7E3', edgecolor='#1F1F1F', linewidth=0.6)
    bars_b = ax.bar(x + width/2, set_b_vals, width, label='Set B (Raw + Engineered)', color='#2F5D8A', edgecolor='#1F1F1F', linewidth=0.6)

    for bar, val in zip(bars_a, set_a_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    for bar, val in zip(bars_b, set_b_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(models_sorted, rotation=25, ha='right', fontsize=11)
    ax.set_ylabel(metric, fontsize=13)
    ax.set_title(f'{metric} Comparison: Set A vs Set B', fontsize=16, fontweight='bold')
    ax.legend(fontsize=11)
    ax.yaxis.grid(True, linestyle=(0, (3, 3)), linewidth=0.7, alpha=0.35)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Feature Set Comparison Across All Models', fontsize=20, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('17_feature_set_comparison.png')
plt.show()
plt.close('all')

Saved figure: ..\outputs\modeling_figures\17_feature_set_comparison.png


## Step 5: Evaluation Metrics Visualization

We visualize model performance using ROC curves, Precision-Recall curves, and confusion matrices. All visualizations use **Feature Set B** (raw + engineered), which represents each model's best version.


In [15]:
model_configs = [
    ('Logistic Regression', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')),
    ('Decision Tree', DecisionTreeClassifier(random_state=42, max_depth=8, min_samples_leaf=10, class_weight='balanced')),
    ('KNN', KNeighborsClassifier(n_neighbors=5)),
    ('SVM', CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=3000, random_state=42))),
    ('Random Forest', RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=200, max_depth=5, min_samples_leaf=10, learning_rate=0.1, random_state=42)),
    ('XGBoost', xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, verbosity=0))
]

colors_models = ['#2F5D8A', '#5B8FA8', '#8C2D2D', '#B85C3A', '#3A7B5C', '#6B4C8A', '#C9A84C']
linestyles = ['-', '--', '-.', ':', '-', '--', '-.']

y_probas_all = {}
roc_aucs = {}

fig, ax = plt.subplots(figsize=(10, 8.5), facecolor='white')

for (name, model), color, ls in zip(model_configs, colors_models, linestyles):
    model.fit(X_b_train, y_train)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_b_test)[:, 1]
    else:
        y_proba = model.decision_function(X_b_test)
    y_probas_all[name] = y_proba

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    roc_aucs[name] = roc_auc

    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=1.8, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], color='#888888', linestyle='--', linewidth=1.0, alpha=0.6, label='Random Classifier')

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=14, labelpad=10)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=14, labelpad=10)

fig.suptitle('ROC Curves — All Models (Feature Set B)', fontsize=20, fontweight='bold', y=0.96)
fig.text(0.5, 0.90, 'Higher AUC indicates better discrimination between default and non-default clients', ha='center', fontsize=11.5, color='#555555')

ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, linestyle=(0, (3, 3)), linewidth=0.7, alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.subplots_adjust(top=0.85, bottom=0.10, left=0.10, right=0.97)
save_fig('19_roc_curves_all_models.png')
plt.show()
plt.close('all')

Saved figure: ..\outputs\modeling_figures\19_roc_curves_all_models.png


In [16]:
fig, ax = plt.subplots(figsize=(10, 8.5), facecolor='white')

baseline = y_test.mean()
ax.axhline(baseline, color='#888888', linestyle='--', linewidth=1.0, alpha=0.6, label=f'No-skill (baseline={baseline:.3f})')

pr_aucs = {}
for (name, _), color, ls in zip(model_configs, colors_models, linestyles):
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_probas_all[name])
    pr_auc = average_precision_score(y_test, y_probas_all[name])
    pr_aucs[name] = pr_auc
    ax.plot(recall_vals, precision_vals, color=color, linestyle=ls, linewidth=1.8, label=f'{name} (PR-AUC={pr_auc:.3f})')

ax.set_xlabel('Recall', fontsize=14, labelpad=10)
ax.set_ylabel('Precision', fontsize=14, labelpad=10)

fig.suptitle('Precision-Recall Curves — All Models (Feature Set B)', fontsize=20, fontweight='bold', y=0.96)
fig.text(0.5, 0.90, 'PR-AUC is more informative than ROC-AUC for imbalanced classification', ha='center', fontsize=11.5, color='#555555')

ax.legend(loc='lower left', fontsize=11, framealpha=0.9)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)
ax.grid(True, linestyle=(0, (3, 3)), linewidth=0.7, alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.subplots_adjust(top=0.85, bottom=0.10, left=0.10, right=0.97)
save_fig('20_pr_curves_all_models.png')
plt.show()
plt.close('all')

Saved figure: ..\outputs\modeling_figures\20_pr_curves_all_models.png


In [17]:
n_models = len(model_configs)
n_cols = 4
n_rows = int(np.ceil(n_models / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows), facecolor='white')
axes = axes.flatten()

for idx, (name, model) in enumerate(model_configs):
    ax = axes[idx]
    y_pred = model.predict(X_b_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-default', 'Default'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.grid(False)

for idx in range(len(model_configs), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Confusion Matrices — All Models (Feature Set B)', fontsize=20, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('21_confusion_matrices_all_models.png')
plt.show()
plt.close('all')

cm_data = []
for name, model in model_configs:
    y_pred = model.predict(X_b_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cm_data.append({'Model': name, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
                    'Default Recall': round(tp / (tp + fn), 4) if (tp+fn) > 0 else 0,
                    'Default Precision': round(tp / (tp + fp), 4) if (tp+fp) > 0 else 0})

cm_df = pd.DataFrame(cm_data)
display(cm_df)
save_csv(cm_df, 'confusion_matrices_detail.csv')

Saved figure: ..\outputs\modeling_figures\21_confusion_matrices_all_models.png


,Model,TN,FP,FN,TP,Default Recall,Default Precision
0,Logistic Regression,2304,2369,371,956,0.7204,0.2875
1,Decision Tree,3511,1162,448,879,0.6624,0.4307
2,KNN,4264,409,1092,235,0.1771,0.3649
3,SVM,4477,196,946,381,0.2871,0.6603
4,Random Forest,4012,661,586,741,0.5584,0.5285
5,Gradient Boosting,4422,251,841,486,0.3662,0.6594
6,XGBoost,3798,875,524,803,0.6051,0.4785


Saved CSV: ..\outputs\results\confusion_matrices_detail.csv


In [18]:
time_data = results_df[results_df['Feature Set'] == 'Set B'][['Model', 'Train Time (s)']].copy()
time_data = time_data.sort_values('Train Time (s)', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5.5), facecolor='white')

colors_time = ['#2F5D8A' if t < 5 else '#8C2D2D' for t in time_data['Train Time (s)']]
bars = ax.barh(time_data['Model'], time_data['Train Time (s)'], color=colors_time, edgecolor='#1F1F1F', linewidth=0.7, height=0.6)

for bar, val in zip(bars, time_data['Train Time (s)']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.2f}s', va='center', fontsize=11, fontweight='bold', color='#1F1F1F')

ax.set_xlabel('Training Time (seconds)', fontsize=13, labelpad=10)
ax.set_xlim(0, time_data['Train Time (s)'].max() * 1.25)

fig.suptitle('Model Training Time Comparison (Feature Set B)', fontsize=18, fontweight='bold', y=0.96)
fig.text(0.5, 0.90, 'SVM and ensemble models require substantially more training time', ha='center', fontsize=11, color='#555555')

ax.xaxis.grid(True, linestyle=(0, (3, 3)), linewidth=0.7, alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.subplots_adjust(left=0.22, right=0.92, top=0.88, bottom=0.10)
save_fig('22_training_time_comparison.png')
plt.show()
plt.close('all')

Saved figure: ..\outputs\modeling_figures\22_training_time_comparison.png


## Step 6: Threshold Optimization

In credit risk prediction, the cost of false negatives (missing a defaulter) can outweigh false positives. We evaluate lower decision thresholds for the top 3 ensemble models to improve recall.


In [19]:
def evaluate_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'Threshold': threshold,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1-score': round(f1_score(y_true, y_pred, zero_division=0), 4)
    }

thresholds_to_test = [0.5, 0.4, 0.35, 0.3, 0.25, 0.2]

top_model_configs = [
    ('Random Forest', RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=200, max_depth=5, min_samples_leaf=10, learning_rate=0.1, random_state=42)),
    ('XGBoost', xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, verbosity=0))
]

all_threshold_results = []
y_probas_top = {}

print('THRESHOLD OPTIMIZATION RESULTS')
print('=' * 80)

for name, model in top_model_configs:
    model.fit(X_b_train, y_train)
    y_proba = model.predict_proba(X_b_test)[:, 1]
    y_probas_top[name] = y_proba

    thr_rows = []
    for t in thresholds_to_test:
        thr_rows.append(evaluate_threshold(y_test, y_proba, t))
    thr_df = pd.DataFrame(thr_rows)
    thr_df.insert(0, 'Model', name)
    all_threshold_results.append(thr_df)

    print(f'\n{name}:')
    display(thr_df)

threshold_summary = pd.concat(all_threshold_results, ignore_index=True)
save_csv(threshold_summary, 'threshold_optimization.csv')

print('\n' + '=' * 60)
print('BEST THRESHOLD PER MODEL (by F1-score)')
print('=' * 60)
for name, _ in top_model_configs:
    model_thr = threshold_summary[threshold_summary['Model'] == name]
    best = model_thr.loc[model_thr['F1-score'].idxmax()]
    print(f'{name:20s} | Best threshold: {best["Threshold"]:.2f} | F1={best["F1-score"]:.4f} | Recall={best["Recall"]:.4f} | Precision={best["Precision"]:.4f}')

THRESHOLD OPTIMIZATION RESULTS

Random Forest:


,Model,Threshold,Accuracy,Precision,Recall,F1-score
0,Random Forest,0.5000,0.7922,0.5285,0.5584,0.5431
1,Random Forest,0.4000,0.7450,0.4471,0.6466,0.5287
2,Random Forest,0.3500,0.6932,0.3924,0.7061,0.5044
3,Random Forest,0.3000,0.6230,0.3461,0.7920,0.4817
4,Random Forest,0.2500,0.5382,0.3088,0.8787,0.4570
5,Random Forest,0.2000,0.4367,0.2727,0.9284,0.4216



Gradient Boosting:


,Model,Threshold,Accuracy,Precision,Recall,F1-score
0,Gradient Boosting,0.5000,0.8180,0.6594,0.3662,0.4709
1,Gradient Boosting,0.4000,0.8120,0.6010,0.4461,0.5121
2,Gradient Boosting,0.3500,0.8017,0.5590,0.4891,0.5217
3,Gradient Boosting,0.3000,0.7907,0.5252,0.5569,0.5406
4,Gradient Boosting,0.2500,0.7687,0.4817,0.6066,0.5370
5,Gradient Boosting,0.2000,0.7297,0.4287,0.6684,0.5224



XGBoost:


,Model,Threshold,Accuracy,Precision,Recall,F1-score
0,XGBoost,0.5000,0.7668,0.4785,0.6051,0.5344
1,XGBoost,0.4000,0.7100,0.4064,0.6760,0.5076
2,XGBoost,0.3500,0.6725,0.3765,0.7332,0.4976
3,XGBoost,0.3000,0.6247,0.3458,0.7815,0.4794
4,XGBoost,0.2500,0.5640,0.3157,0.8320,0.4577
5,XGBoost,0.2000,0.5055,0.2948,0.8877,0.4426


Saved CSV: ..\outputs\results\threshold_optimization.csv

BEST THRESHOLD PER MODEL (by F1-score)
Random Forest        | Best threshold: 0.50 | F1=0.5431 | Recall=0.5584 | Precision=0.5285
Gradient Boosting    | Best threshold: 0.30 | F1=0.5406 | Recall=0.5569 | Precision=0.5252
XGBoost              | Best threshold: 0.50 | F1=0.5344 | Recall=0.6051 | Precision=0.4785


In [20]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='white')
colors_top = ['#2F5D8A', '#5B8FA8', '#8C2D2D']

for ax_idx, (name, _) in enumerate(top_model_configs):
    ax = axes[ax_idx]
    model_data = threshold_summary[threshold_summary['Model'] == name]

    ax.plot(model_data['Threshold'], model_data['Precision'], marker='o', color='#2F5D8A', linewidth=1.8, markersize=7, label='Precision')
    ax.plot(model_data['Threshold'], model_data['Recall'], marker='s', color='#8C2D2D', linewidth=1.8, markersize=7, label='Recall')
    ax.plot(model_data['Threshold'], model_data['F1-score'], marker='^', color='#3A7B5C', linewidth=2.2, markersize=8, label='F1-score')

    best_idx = model_data['F1-score'].idxmax()
    best_thr = model_data.loc[best_idx, 'Threshold']
    best_f1 = model_data.loc[best_idx, 'F1-score']

    ax.axvline(best_thr, color='#888888', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.annotate(f'Best F1={best_f1:.3f}\nat thr={best_thr:.2f}',
                xy=(best_thr, best_f1), xytext=(best_thr + 0.06, best_f1 - 0.08),
                fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#444444', linewidth=1.2))

    ax.set_xlabel('Decision Threshold', fontsize=13, labelpad=8)
    ax.set_ylabel('Score', fontsize=13, labelpad=8)
    ax.set_title(f'{name}', fontsize=16, fontweight='bold')
    ax.set_xlim(0.18, 0.52)
    ax.set_ylim(0.2, 1.0)
    ax.legend(fontsize=11, loc='upper right')
    ax.grid(True, linestyle=(0, (3, 3)), linewidth=0.7, alpha=0.35)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Threshold Optimization — Precision / Recall / F1 Trade-off', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('25_threshold_optimization.png')
plt.show()
plt.close('all')

Saved figure: ..\outputs\modeling_figures\25_threshold_optimization.png


## Step 7: Interpretability

We examine feature importance for tree-based models and coefficient analysis for logistic regression. This helps identify which features drive default predictions.


In [21]:
tree_models_list = [
    ('Random Forest', trained_models[('Random Forest', 'Set B')]),
    ('Gradient Boosting', trained_models[('Gradient Boosting', 'Set B')]),
    ('XGBoost', trained_models[('XGBoost', 'Set B')])
]

fig, axes = plt.subplots(1, 3, figsize=(21, 8), facecolor='white')
top_n_features = 15

for ax_idx, (name, model) in enumerate(tree_models_list):
    ax = axes[ax_idx]
    importances = model.feature_importances_
    feat_names = feature_set_b
    sorted_idx = np.argsort(importances)[::-1][:top_n_features]
    top_importances = importances[sorted_idx][::-1]
    top_names = [feat_names[i] for i in sorted_idx][::-1]

    colors_imp = ['#2F5D8A'] * len(top_names)
    for i in range(min(3, len(top_names))):
        colors_imp[-(i + 1)] = '#8C2D2D'

    bars = ax.barh(range(len(top_names)), top_importances, color=colors_imp, edgecolor='#1F1F1F', linewidth=0.6, height=0.65)
    ax.set_yticks(range(len(top_names)))
    ax.set_yticklabels(top_names, fontsize=11)
    ax.set_xlabel('Feature Importance', fontsize=13)
    ax.set_title(f'{name} — Top {top_n_features} Features', fontsize=15, fontweight='bold')
    ax.xaxis.grid(True, linestyle=(0, (3, 3)), linewidth=0.7, alpha=0.35)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for bar, val in zip(bars, top_importances):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8.5)

fig.suptitle('Feature Importance — Tree-based Models (Feature Set B)', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('27_feature_importance_tree_models.png')
plt.show()
plt.close('all')

importance_rows = []
for name, model in tree_models_list:
    for feat, imp in zip(feature_set_b, model.feature_importances_):
        importance_rows.append({'Model': name, 'Feature': feat, 'Importance': round(imp, 6)})
importance_df = pd.DataFrame(importance_rows)
importance_df = importance_df.sort_values(['Model', 'Importance'], ascending=[True, False])
save_csv(importance_df, 'feature_importance_tree_models.csv')

print('\nTop 10 features across all tree models (mean importance):')
mean_imp = importance_df.groupby('Feature')['Importance'].mean().sort_values(ascending=False).head(10)
display(mean_imp.to_frame('Mean Importance'))

Saved figure: ..\outputs\modeling_figures\27_feature_importance_tree_models.png
Saved CSV: ..\outputs\results\feature_importance_tree_models.csv

Top 10 features across all tree models (mean importance):


,Mean Importance
Feature,
max_delay,0.1072
delay_count,0.1054
recent_delay,0.1024
PAY_0,0.0874
severe_delay_count,0.0856
credit_utilization,0.0278
avg_delay,0.0245
avg_pay_amt,0.0240
bill_trend,0.0221


In [22]:
lr_model_b = trained_models[('Logistic Regression', 'Set B')]

coefficients = pd.DataFrame({
    'Feature': feature_set_b,
    'Coefficient': lr_model_b.coef_[0],
    'Abs Coefficient': np.abs(lr_model_b.coef_[0])
})
coefficients = coefficients.sort_values('Abs Coefficient', ascending=False)

print('Logistic Regression — Top 15 Coefficients (Set B)')
display(coefficients.head(15))
save_csv(coefficients, 'logistic_regression_coefficients.csv')

top_n_coef = 15
top_coef = coefficients.head(top_n_coef).sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(10, 7), facecolor='white')

colors_coef = ['#8C2D2D' if c > 0 else '#2F5D8A' for c in top_coef['Coefficient']]
bars = ax.barh(range(len(top_coef)), top_coef['Coefficient'], color=colors_coef, edgecolor='#1F1F1F', linewidth=0.6, height=0.65)
ax.axvline(0, color='#444444', linewidth=1.0)

for bar, val in zip(bars, top_coef['Coefficient']):
    x_offset = 0.01 if val >= 0 else -0.01
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + x_offset, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', ha=ha, fontsize=9, fontweight='bold')

ax.set_yticks(range(len(top_coef)))
ax.set_yticklabels(top_coef['Feature'], fontsize=11)
ax.set_xlabel('Coefficient Value', fontsize=13)

fig.suptitle('Logistic Regression Coefficients (Feature Set B)', fontsize=18, fontweight='bold', y=0.96)
fig.text(0.5, 0.905, 'Positive coefficients increase default risk; negative coefficients decrease it', ha='center', fontsize=11.5, color='#555555')

ax.xaxis.grid(True, linestyle=(0, (3, 3)), linewidth=0.7, alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.subplots_adjust(left=0.28, right=0.93, top=0.85, bottom=0.10)
save_fig('28_logistic_regression_coefficients.png')
plt.show()
plt.close('all')

print('\nKey features:')
for feat in ['PAY_0', 'delay_count', 'severe_delay_count', 'LIMIT_BAL', 'credit_utilization']:
    coef_val = coefficients.loc[coefficients['Feature'] == feat, 'Coefficient'].values[0]
    direction = 'increases' if coef_val > 0 else 'decreases'
    print(f'  {feat:25s} coef={coef_val:+.4f} ({direction} default risk)')

Logistic Regression — Top 15 Coefficients (Set B)


,Feature,Coefficient,Abs Coefficient
24,severe_delay_count,2.2403,2.2403
23,delay_count,-1.8418,1.8418
28,recent_delay_flag,0.8627,0.8627
35,max_pay_amt,0.5296,0.5296
25,max_delay,0.3784,0.3784
30,max_bill_amt,-0.3447,0.3447
0,LIMIT_BAL,-0.2381,0.2381
34,avg_pay_amt,-0.2337,0.2337
18,PAY_AMT2,-0.2319,0.2319
27,recent_delay,-0.1691,0.1691


Saved CSV: ..\outputs\results\logistic_regression_coefficients.csv
Saved figure: ..\outputs\modeling_figures\28_logistic_regression_coefficients.png

Key features:
  PAY_0                     coef=-0.1691 (decreases default risk)
  delay_count               coef=-1.8418 (decreases default risk)
  severe_delay_count        coef=+2.2403 (increases default risk)
  LIMIT_BAL                 coef=-0.2381 (decreases default risk)
  credit_utilization        coef=+0.0987 (increases default risk)


## Step 8: Summary and Conclusions

Here we present the final model ranking and key findings from the modeling phase.


In [29]:
final_results = results_df[results_df['Feature Set'] == 'Set B'].copy()
final_display = final_results[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'PR-AUC', 'Train Time (s)']].copy()
final_display = final_display.sort_values('F1-score', ascending=False).reset_index(drop=True)
final_display.insert(0, 'Rank', range(1, len(final_display) + 1))

print('=' * 120)
print('FINAL MODEL RANKING (Feature Set B)')
print('=' * 120)
display(final_display)
save_csv(final_display, 'final_model_ranking.csv')

best_model_row = final_display.iloc[0]
print(f'\nBest overall model: {best_model_row["Model"]}')
print(f'  F1-score:  {best_model_row["F1-score"]}')
print(f'  ROC-AUC:   {best_model_row["ROC-AUC"]}')
print(f'  PR-AUC:    {best_model_row["PR-AUC"]}')

print('\n' + '=' * 60)
print('DEFINITIVE FINDINGS')
print('=' * 60)
print('1. ENGINEERED FEATURES: Feature Set B outperforms Set A across most models,')
print('   confirming that repayment-behavior features add predictive value.')
print(f'2. BEST MODEL: {best_model_row["Model"]} achieves the highest F1-score ({best_model_row["F1-score"]})')
print(f'   and ROC-AUC ({best_model_row["ROC-AUC"]}) on the hold-out test set.')
print('3. KEY PREDICTORS: PAY_0, delay_count, severe_delay_count, LIMIT_BAL,')
print('   and credit_utilization are the most important features.')
print('4. THRESHOLD SENSITIVITY: Lowering the decision threshold from 0.5 to 0.3-0.35')
print('   improves recall substantially with manageable precision loss.')

FINAL MODEL RANKING (Feature Set B)


,Rank,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC,Train Time (s)
0,1,Random Forest,0.7922,0.5285,0.5584,0.5431,0.7785,0.5545,1.0900
1,2,XGBoost,0.7668,0.4785,0.6051,0.5344,0.7722,0.5465,0.4570
2,3,Decision Tree,0.7317,0.4307,0.6624,0.5220,0.7646,0.5288,0.3080
3,4,Logistic Regression,0.7558,0.4599,0.5968,0.5195,0.7583,0.5243,0.0530
4,5,Gradient Boosting,0.8180,0.6594,0.3662,0.4709,0.7778,0.5507,40.7520
5,6,SVM,0.8162,0.6564,0.3542,0.4601,0.7559,0.5173,0.6570
6,7,KNN,0.7928,0.5473,0.3662,0.4388,0.7075,0.4222,0.0060


Saved CSV: ..\outputs\results\final_model_ranking.csv

Best overall model: Random Forest
  F1-score:  0.5431
  ROC-AUC:   0.7785
  PR-AUC:    0.5545

DEFINITIVE FINDINGS
1. ENGINEERED FEATURES: Feature Set B outperforms Set A across most models,
   confirming that repayment-behavior features add predictive value.
2. BEST MODEL: Random Forest achieves the highest F1-score (0.5431)
   and ROC-AUC (0.7785) on the hold-out test set.
3. KEY PREDICTORS: PAY_0, delay_count, severe_delay_count, LIMIT_BAL,
   and credit_utilization are the most important features.
4. THRESHOLD SENSITIVITY: Lowering the decision threshold from 0.5 to 0.3-0.35
   improves recall substantially with manageable precision loss.


In [30]:
# Save probabilities for top models
proba_records = []
for name, _ in top_model_configs:
    y_proba = y_probas_top[name]
    for true_val, proba_val in zip(y_test.values, y_proba):
        proba_records.append({'Model': name, 'True': true_val, 'Predicted_Proba': proba_val})
proba_df = pd.DataFrame(proba_records)
save_csv(proba_df, 'model_probabilities.csv')

# Save LaTeX table
#latex_table = final_display.to_latex(index=False, float_format='%.4f')
#with open(RESULT_DIR / 'final_model_ranking.tex', 'w', encoding='utf-8') as f:
#    f.write(latex_table)
#print('LaTeX table saved.')

# Summary of all outputs
print('\n' + '=' * 60)
print('ALL SAVED OUTPUTS')
print('=' * 60)
print('\nCSV files in', RESULT_DIR)
for f in sorted(RESULT_DIR.glob('*.csv')):
    print(f'  {f.name}')
print('\nFigure files in', FIG_DIR)
for f in sorted(FIG_DIR.glob('*.png')):
    print(f'  {f.name}')
print('\nModeling notebook complete.')

Saved CSV: ..\outputs\results\model_probabilities.csv

ALL SAVED OUTPUTS

CSV files in ..\outputs\results
  all_models_comparison.csv
  confusion_matrices_detail.csv
  decision_tree_results.csv
  education_after_fix.csv
  feature_importance_tree_models.csv
  feature_set_comparison.csv
  final_model_ranking.csv
  gradient_boosting_results.csv
  knn_results.csv
  logistic_regression_coefficients.csv
  logistic_regression_results.csv
  marriage_after_fix.csv
  model_probabilities.csv
  random_forest_results.csv
  svm_results.csv
  threshold_optimization.csv
  xgboost_results.csv

Figure files in ..\outputs\modeling_figures
  17_feature_set_comparison.png
  19_roc_curves_all_models.png
  20_pr_curves_all_models.png
  21_confusion_matrices_all_models.png
  22_training_time_comparison.png
  25_threshold_optimization.png
  27_feature_importance_tree_models.png
  28_logistic_regression_coefficients.png
  29_dnn_learning_curves.png
  30_ultimate_ensemble_roc.png

Modeling notebook complete.


# Advanced Modeling Stage: Deep Neural Network (DNN) Implementation

## 1. Structural Overview
This module transitions the credit card default prediction task from traditional machine learning frameworks into deep learning. Built natively on the PyTorch ecosystem, the code implements a fully connected feedforward Deep Neural Network (DNN) designed to extract high-order, non-linear interactions among customer repayment behaviors. 

## 2. Technical Component Breakdown

### 2.1 Hardware and Runtime Alignment
The script initiates by dynamically detecting GPU availability via `torch.device('cuda')`. It falls back gracefully to standard execution pipelines if hardware-accelerated environments are not present, ensuring reliable compute resource utilization.

### 2.2 Tensor Pipelines and Mini-batch Loading
Data frames derived from the upstream feature engineering architecture are extracted into NumPy arrays and wrapped in `torch.FloatTensor` structures. 
* To prevent target distribution shift during evaluations, a stratifying train/validation partition ratio of `85:15` is executed via `train_test_split`.
* Mini-batch processing is handled via `DataLoader` with an optimal batch scale of 512. Shuffling is exclusively assigned to the training loader to optimize gradient stochasticity while preserving validation sequence integrity.

### 2.3 Regularized Architecture Design
The network inherits from `nn.Module` and cascades across sequential hidden layers (`Input -> 128 -> 64 -> 32 -> 1`). To stabilize convergence against internal covariate shifts common in complex tabular finance datasets, `nn.BatchNorm1d` is appended behind each linear layer. `nn.Dropout(0.3)` is selectively placed across high-dimensional features to minimize co-adaptation risks and prevent structural overfitting.

### 2.4 Numerically Stable Loss & Optimization
Optimization relies on the `Adam` variant backed by an initialized learning rate of `0.001` and an L2 penalty weight decay parameter of `1e-5`. For classification cost analysis, `nn.BCEWithLogitsLoss` is leveraged. By processing raw network logits prior to sigmoid scaling, it avoids logarithmic boundary constraints, securing optimal numerical stability during error backpropagation.

### 2.5 Validation Intercept & Early Stopping
The training epoch horizon is capped at 150 cycles. To protect structural generalization thresholds, an early stopping mechanism evaluates validation losses at every epoch completion. If validation loss stagnation persists beyond a `patience` configuration of 15 iterations, execution halts prematurely, and optimal weight states are restored via serialized local storage buffers (`best_dnn_model.pth`).

In [31]:
# ==========================================
# ADVANCED MODEL: Deep Neural Network (DNN - PyTorch)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import numpy as np
import time

print('=' * 80)
print('ADVANCED MODEL: DEEP NEURAL NETWORK (DNN - PyTorch)')
print('=' * 80)

# 1. Check GPU Availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

# 2. Data Preparation: Convert to PyTorch Tensors
# Extract NumPy arrays from teammate's engineered feature set (Set B)
X = X_b_train_s.values
y = y_train.values.astype(np.float32)
X_test_dnn = X_b_test_s.values
y_test_dnn = y_test.values.astype(np.float32)

# Split a validation set for deep learning monitoring and early stopping
X_train_dnn, X_val_dnn, y_train_dnn, y_val_dnn = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Convert to Tensors
X_train_t = torch.FloatTensor(X_train_dnn)
y_train_t = torch.FloatTensor(y_train_dnn).unsqueeze(1)
X_val_t = torch.FloatTensor(X_val_dnn)
y_val_t = torch.FloatTensor(y_val_dnn).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test_dnn)

# Build DataLoaders for batch training
batch_size = 512
train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# 3. Define DNN Architecture
class CreditRiskDNN(nn.Module):
    def __init__(self, input_dim):
        super(CreditRiskDNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),  # Prevent overfitting
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 1)  # Output layer; no Sigmoid needed when combined with BCEWithLogitsLoss
        )
        
    def forward(self, x):
        return self.network(x)

input_dim = X_train_t.shape[1]
model = CreditRiskDNN(input_dim).to(device)

# 4. Define Loss Function and Optimizer
# BCEWithLogitsLoss integrates Sigmoid internally for better numerical stability
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# 5. Model Training and Early Stopping Mechanism
epochs = 150
patience = 15
best_val_loss = float('inf')
counter = 0

train_losses, val_losses = [], []
val_aucs = []

print("\nStarting Training...")
t0 = time.time()

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    all_val_preds = []
    all_val_labels = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            
            # Record predicted probabilities for AUC calculation
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_val_preds.extend(probs)
            all_val_labels.extend(labels.cpu().numpy())
            
    epoch_val_loss = val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)
    
    val_auc = roc_auc_score(all_val_labels, all_val_preds)
    val_aucs.append(val_auc)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val AUC: {val_auc:.4f}")
        
    # Early stopping logic
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        counter = 0
        # Save the best model weights
        torch.save(model.state_dict(), 'best_dnn_model.pth')
    else:
        counter += 1
        if counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}!")
            break

train_time_dnn = time.time() - t0
print(f"\nDNN Training Time: {train_time_dnn:.2f} seconds")

# 6. Final Test Set Evaluation
# Load the best weights
model.load_state_dict(torch.load('best_dnn_model.pth'))
model.eval()

with torch.no_grad():
    X_test_tensor = X_test_t.to(device)
    test_outputs = model(X_test_tensor)
    # Convert Logits to probabilities
    y_proba_dnn = torch.sigmoid(test_outputs).cpu().numpy().flatten()

# Convert probabilities to classes (Threshold = 0.5)
y_pred_dnn = (y_proba_dnn >= 0.5).astype(int)

# Calculate metrics to align with teammate's code
acc_dnn = accuracy_score(y_test_dnn, y_pred_dnn)
prec_dnn = precision_score(y_test_dnn, y_pred_dnn)
rec_dnn = recall_score(y_test_dnn, y_pred_dnn)
f1_dnn = f1_score(y_test_dnn, y_pred_dnn)
roc_auc_dnn = roc_auc_score(y_test_dnn, y_proba_dnn)
pr_auc_dnn = average_precision_score(y_test_dnn, y_proba_dnn)

print('\n' + '=' * 40)
print('DNN FINAL PERFORMANCE (Test Set)')
print('=' * 40)
print(f"Accuracy:  {acc_dnn:.4f}")
print(f"Precision: {prec_dnn:.4f}")
print(f"Recall:    {rec_dnn:.4f}")
print(f"F1-score:  {f1_dnn:.4f}")
print(f"ROC-AUC:   {roc_auc_dnn:.4f}")
print(f"PR-AUC:    {pr_auc_dnn:.4f}")

# 7. Plot Learning Curves (Perfect for Slide 11)
fig, ax1 = plt.subplots(figsize=(10, 6), facecolor='white')

ax1.plot(train_losses, label='Train Loss', color='#2F5D8A', linewidth=2)
ax1.plot(val_losses, label='Validation Loss', color='#8C2D2D', linewidth=2)
ax1.set_xlabel('Epochs', fontsize=14)
ax1.set_ylabel('Loss (Binary Cross Entropy)', fontsize=14)
ax1.tick_params(axis='y')
ax1.legend(loc='upper left', fontsize=12)

ax2 = ax1.twinx()
ax2.plot(val_aucs, label='Validation AUC', color='#3A7B5C', linewidth=2, linestyle='--')
ax2.set_ylabel('ROC AUC', fontsize=14)
ax2.tick_params(axis='y')
ax2.legend(loc='upper right', fontsize=12)

plt.title('DNN Learning Curves: Loss & AUC', fontsize=16, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
save_fig('29_dnn_learning_curves.png')
plt.show()

ADVANCED MODEL: DEEP NEURAL NETWORK (DNN - PyTorch)
Training on device: cpu

Starting Training...
Epoch [10/150] | Train Loss: 0.4290 | Val Loss: 0.4332 | Val AUC: 0.7780
Epoch [20/150] | Train Loss: 0.4245 | Val Loss: 0.4333 | Val AUC: 0.7802
Epoch [30/150] | Train Loss: 0.4200 | Val Loss: 0.4345 | Val AUC: 0.7784

Early stopping triggered at epoch 31!

DNN Training Time: 10.42 seconds

DNN FINAL PERFORMANCE (Test Set)
Accuracy:  0.8190
Precision: 0.6609
Recall:    0.3730
F1-score:  0.4769
ROC-AUC:   0.7740
PR-AUC:    0.5589
Saved figure: ..\outputs\modeling_figures\29_dnn_learning_curves.png


### 1. Confusion Matrix Heatmap

**Description:**
This code snippet generates a styled Confusion Matrix Heatmap using `seaborn` and `matplotlib`. It provides a direct visual breakdown of the model's classification performance at the chosen threshold (default `0.5`). 

**Business Value:**
By displaying the exact counts of True Positives (correctly identified defaults), True Negatives (correctly identified safe customers), False Positives (false alarms), and False Negatives (missed defaults), this plot helps stakeholders instantly grasp the financial impact of the model's decision boundaries.

In [26]:
# ==========================================
# EXTRA PLOT 1: Confusion Matrix Heatmap
# ==========================================
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test_dnn, y_pred_dnn)

plt.figure(figsize=(8, 6), facecolor='white')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'],
            annot_kws={'size': 16, 'weight': 'bold'})

plt.title('Confusion Matrix: Financial Risk Breakdown', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=14, labelpad=10)
plt.ylabel('True Label', fontsize=14, labelpad=10)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.savefig('dnn_confusion_matrix.png', dpi=300, bbox_inches='tight')
print("Image saved as: dnn_confusion_matrix.png")
plt.show()

Image saved as: dnn_confusion_matrix.png


### 2. Dual-Axis Evaluation: ROC and Precision-Recall Curves

**Description:**
This visualization renders a dual-subplot figure containing both the Receiver Operating Characteristic (ROC) curve and the Precision-Recall (PR) curve side-by-side. 

**Business Value:**
While the ROC-AUC score measures the overall ranking capability of the model, it can sometimes present an overly optimistic view on highly imbalanced credit risk data. The inclusion of the PR Curve (and PR-AUC) provides a much stricter and more reliable evaluation of the model's ability to precisely capture the minority class (defaulters) without generating excessive false alarms.

In [27]:
# ==========================================
# EXTRA PLOT 2: Twin ROC and Precision-Recall Curves
# ==========================================
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(y_test_dnn, y_proba_dnn)
precision, recall, _ = precision_recall_curve(y_test_dnn, y_proba_dnn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), facecolor='white')

ax1.plot(fpr, tpr, color='#2F5D8A', linewidth=3, label=f'DNN (AUC = {roc_auc_dnn:.4f})')
ax1.plot([0, 1], [0, 1], color='#888888', linestyle='--', alpha=0.7)
ax1.set_title('Receiver Operating Characteristic (ROC)', fontsize=15, fontweight='bold', pad=15)
ax1.set_xlabel('False Positive Rate', fontsize=13)
ax1.set_ylabel('True Positive Rate', fontsize=13)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower right', fontsize=12)

ax2.plot(recall, precision, color='#8C2D2D', linewidth=3, label=f'DNN (PR-AUC = {pr_auc_dnn:.4f})')
ax2.set_title('Precision-Recall (PR) Curve', fontsize=15, fontweight='bold', pad=15)
ax2.set_xlabel('Recall', fontsize=13)
ax2.set_ylabel('Precision', fontsize=13)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='lower left', fontsize=12)

plt.suptitle('DNN Discriminative Power Analysis', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()

plt.savefig('dnn_evaluation_curves.png', dpi=300, bbox_inches='tight')
print("Image saved as: dnn_evaluation_curves.png")
plt.show()

Image saved as: dnn_evaluation_curves.png


### 3. Prediction Probability Distribution Density

**Description:**
This code plots the probability density histograms of the neural network's raw output (before applying a hard threshold). It separates the distribution into two distinct groups based on the actual true labels: True Non-Default (Safe) vs. True Default (Risky).

**Business Value:**
This density plot acts as an "X-ray" of the model's internal confidence. A strong model will show two clearly separated peaks (one near 0.0, one near 1.0). Overlapping areas represent the "grey zone" where the model is uncertain. This visualization is crucial for explaining to risk managers exactly *why* a specific decision threshold (e.g., shifting from 0.5 to 0.4) was chosen to maximize business revenue.

In [28]:
# ==========================================
# EXTRA PLOT 2: Twin ROC and Precision-Recall Curves
# ==========================================
prob_non_default = y_proba_dnn[y_test_dnn == 0]
prob_default = y_proba_dnn[y_test_dnn == 1]

plt.figure(figsize=(10, 6), facecolor='white')

plt.hist(prob_non_default, bins=50, alpha=0.6, density=True, 
         color='#2F5D8A', label='True Non-Default (Safe)', edgecolor='white')
plt.hist(prob_default, bins=50, alpha=0.6, density=True, 
         color='#8C2D2D', label='True Default (High Risk)', edgecolor='white')

plt.title('Prediction Probability Distribution by Class', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted Probability of Default (Output of Sigmoid)', fontsize=14)
plt.ylabel('Density', fontsize=14)
plt.xlim(0, 1)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=12)

plt.savefig('dnn_probability_density.png', dpi=300, bbox_inches='tight')
print("Image saved as: dnn_probability_density.png")
plt.tight_layout()
plt.show()

Image saved as: dnn_probability_density.png
